<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/04_Foundations/mathematics/04_probability_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Probability Fundamentals

Probability is the language of uncertainty — and machine learning is fundamentally about making predictions under uncertainty. This notebook covers everything from basic axioms to Maximum Likelihood Estimation.

**Topics:**
- Probability axioms and rules
- Conditional probability and Bayes' theorem
- Key distributions (discrete and continuous)
- Expectation, variance, covariance
- Maximum Likelihood Estimation (MLE)
- Law of Large Numbers and Central Limit Theorem preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

## 1. Probability Axioms and Rules

In [ ]:
# Simulate: roll a fair 6-sided die 100,000 times
n = 100_000
rolls = np.random.randint(1, 7, size=n)

# Empirical probabilities converge to true probabilities
for face in range(1, 7):
    empirical = (rolls == face).mean()
    print(f'P(die={face}) empirical={empirical:.4f}, true=1/6={1/6:.4f}')

print('\n--- Probability Rules ---')
# Addition rule: P(A or B) = P(A) + P(B) - P(A and B)
P_even = (rolls % 2 == 0).mean()       # P(even): 2,4,6
P_gt3  = (rolls > 3).mean()            # P(>3): 4,5,6
P_even_and_gt3 = ((rolls % 2 == 0) & (rolls > 3)).mean()  # 4,6
P_even_or_gt3  = ((rolls % 2 == 0) | (rolls > 3)).mean()  # 2,4,5,6

print(f'P(even)       = {P_even:.4f} (true: 3/6={3/6:.4f})')
print(f'P(>3)         = {P_gt3:.4f} (true: 3/6={3/6:.4f})')
print(f'P(even AND >3)= {P_even_and_gt3:.4f} (true: 2/6={2/6:.4f})')
print(f'P(even OR >3) = {P_even_or_gt3:.4f} (true: 4/6={4/6:.4f})')
print(f'Addition rule check: P(even)+P(>3)-P(both) = {P_even+P_gt3-P_even_and_gt3:.4f}')

## 2. Bayes' Theorem

**P(A|B) = P(B|A) · P(A) / P(B)**

This is the foundation of Bayesian ML, spam filters, medical diagnosis, and more.

In [ ]:
# Classic example: Medical test
# Disease prevalence: 1% of population
# Test sensitivity (true positive rate): 99%
# Test specificity (true negative rate): 95%
# Question: If you test positive, what's the probability you have the disease?

P_disease = 0.01        # prior: P(disease)
P_pos_given_disease = 0.99  # likelihood: P(+|disease)
P_pos_given_healthy = 0.05  # false positive rate: P(+|no disease)
P_healthy = 1 - P_disease

# Total probability: P(+) = P(+|disease)·P(disease) + P(+|healthy)·P(healthy)
P_positive = P_pos_given_disease * P_disease + P_pos_given_healthy * P_healthy

# Bayes: P(disease|+) = P(+|disease) · P(disease) / P(+)
P_disease_given_pos = (P_pos_given_disease * P_disease) / P_positive

print('Medical Test Example')
print(f'  Disease prevalence: {P_disease:.0%}')
print(f'  Test sensitivity (TPR): {P_pos_given_disease:.0%}')
print(f'  Test specificity (TNR): {1-P_pos_given_healthy:.0%}')
print(f'  P(positive test) = {P_positive:.4f}')
print(f'  P(disease | positive test) = {P_disease_given_pos:.2%}')
print(f'\n  → Even with a 99% sensitive test, testing positive only means')
print(f'    ~{P_disease_given_pos:.0%} chance of actually having the disease!')
print(f'    (because the disease is rare — base rate matters!)')

# Visualize how posterior changes with different priors
priors = np.linspace(0.001, 0.5, 200)
posteriors = (P_pos_given_disease * priors) / (P_pos_given_disease * priors + P_pos_given_healthy * (1 - priors))

plt.figure(figsize=(8, 5))
plt.plot(priors * 100, posteriors * 100, 'b-', lw=2)
plt.axvline(P_disease * 100, color='red', ls='--', label=f'Current prior ({P_disease:.0%})')
plt.axhline(P_disease_given_pos * 100, color='green', ls='--', 
            label=f'Current posterior ({P_disease_given_pos:.1%})')
plt.xlabel('Prior P(disease) %'); plt.ylabel('Posterior P(disease|+) %')
plt.title("Bayes' Theorem: How Prior Affects Posterior")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Key Probability Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# 1. Bernoulli: coin flip, p=0.7
ax = axes[0, 0]
for p, color in [(0.3, 'steelblue'), (0.5, 'darkorange'), (0.7, 'green')]:
    samples = np.random.binomial(1, p, 10000)
    ax.bar([0, 1], [(samples==0).mean(), (samples==1).mean()], alpha=0.5, width=0.3,
           label=f'p={p}', bottom=0)
ax.set_title('Bernoulli (coin flip)'); ax.set_xlabel('Outcome'); ax.set_ylabel('Probability')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Tails (0)', 'Heads (1)'])
ax.legend(); ax.grid(True, alpha=0.3)

# 2. Binomial: n=20 trials
ax = axes[0, 1]
n_trials = 20
for p, color in [(0.3, 'steelblue'), (0.5, 'darkorange'), (0.8, 'green')]:
    k = np.arange(0, n_trials+1)
    pmf = stats.binom.pmf(k, n=n_trials, p=p)
    ax.plot(k, pmf, 'o-', color=color, lw=1.5, markersize=4, label=f'p={p}')
ax.set_title(f'Binomial (n={n_trials})'); ax.set_xlabel('k successes')
ax.set_ylabel('P(X=k)'); ax.legend(); ax.grid(True, alpha=0.3)

# 3. Poisson: events per interval
ax = axes[0, 2]
k = np.arange(0, 20)
for lam, color in [(2, 'steelblue'), (5, 'darkorange'), (10, 'green')]:
    pmf = stats.poisson.pmf(k, mu=lam)
    ax.plot(k, pmf, 'o-', color=color, lw=1.5, markersize=4, label=f'λ={lam}')
ax.set_title('Poisson (rare events)'); ax.set_xlabel('k')
ax.set_ylabel('P(X=k)'); ax.legend(); ax.grid(True, alpha=0.3)

# 4. Normal (Gaussian)
ax = axes[1, 0]
x = np.linspace(-6, 6, 300)
for (mu, sigma), color in [((0, 1), 'steelblue'), ((0, 2), 'darkorange'), ((2, 0.5), 'green')]:
    ax.plot(x, stats.norm.pdf(x, mu, sigma), color=color, lw=2, label=f'μ={mu}, σ={sigma}')
ax.set_title('Normal (Gaussian)'); ax.set_xlabel('x'); ax.set_ylabel('PDF')
ax.legend(); ax.grid(True, alpha=0.3)

# 5. Exponential
ax = axes[1, 1]
x = np.linspace(0, 5, 300)
for lam, color in [(0.5, 'steelblue'), (1, 'darkorange'), (2, 'green')]:
    ax.plot(x, stats.expon.pdf(x, scale=1/lam), color=color, lw=2, label=f'λ={lam}')
ax.set_title('Exponential (time to event)'); ax.set_xlabel('x')
ax.set_ylabel('PDF'); ax.legend(); ax.grid(True, alpha=0.3)

# 6. Beta distribution (used in Bayesian inference)
ax = axes[1, 2]
x = np.linspace(0, 1, 300)
for (a, b), color, label in [((1,1), 'steelblue', 'Beta(1,1)=Uniform'), 
                               ((2,5), 'darkorange', 'Beta(2,5)'),
                               ((8,2), 'green', 'Beta(8,2)'),
                               ((2,2), 'red', 'Beta(2,2)')]:
    ax.plot(x, stats.beta.pdf(x, a, b), color=color, lw=2, label=label)
ax.set_title('Beta (prior for probabilities)'); ax.set_xlabel('x ∈ [0,1]')
ax.set_ylabel('PDF'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Key Probability Distributions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Expectation, Variance & Covariance

In [ ]:
# Generate correlated data
mean = [0, 0]
cov_matrix = [[1, 0.8], [0.8, 1]]  # strong positive correlation
data = np.random.multivariate_normal(mean, cov_matrix, size=1000)
X, Y = data[:, 0], data[:, 1]

# Expectation (mean)
E_X = X.mean()
E_Y = Y.mean()
print(f'E[X] = {E_X:.3f}, E[Y] = {E_Y:.3f}')

# Variance: Var(X) = E[(X - E[X])^2]
Var_X = ((X - E_X)**2).mean()
print(f'Var(X) = {Var_X:.3f}, Std(X) = {np.sqrt(Var_X):.3f}')

# Covariance: Cov(X,Y) = E[(X - E[X])(Y - E[Y])]
Cov_XY = ((X - E_X) * (Y - E_Y)).mean()
print(f'Cov(X,Y) = {Cov_XY:.3f}')

# Pearson correlation = Cov(X,Y) / (Std(X) * Std(Y))
rho = Cov_XY / (np.std(X) * np.std(Y))
print(f'Correlation ρ = {rho:.3f} (true: 0.8)')

# Full covariance matrix
cov_empirical = np.cov(data.T)
print(f'\nEmpirical covariance matrix:\n{cov_empirical.round(3)}')
print(f'True covariance matrix:\n{np.array(cov_matrix)}')

plt.figure(figsize=(7, 5))
plt.scatter(X, Y, alpha=0.3, s=10)
plt.xlabel('X'); plt.ylabel('Y')
plt.title(f'Correlated Data: ρ = {rho:.2f}')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Maximum Likelihood Estimation (MLE)

MLE: find parameters θ that maximize the probability of observing the data.

**L(θ) = P(data | θ)** — maximize this over θ.

In [ ]:
# MLE for a Normal distribution: given data, find the best μ and σ
# True parameters
true_mu, true_sigma = 5.0, 2.0
n_samples = 200
data = np.random.normal(true_mu, true_sigma, n_samples)

# MLE for Normal:
# μ_MLE = sample mean
# σ_MLE = sample standard deviation (biased; use n, not n-1)
mu_mle = data.mean()
sigma_mle = data.std()  # MLE is biased (divides by n, not n-1)
sigma_unbiased = data.std(ddof=1)  # Unbiased estimator (divides by n-1)

print(f'True params:      μ={true_mu}, σ={true_sigma}')
print(f'MLE estimates:    μ̂={mu_mle:.3f}, σ̂={sigma_mle:.3f} (biased)')
print(f'Unbiased σ̂:      {sigma_unbiased:.3f}')

# Visualize log-likelihood surface
mus = np.linspace(3, 7, 100)
sigmas = np.linspace(0.5, 4, 100)
MU, SIGMA = np.meshgrid(mus, sigmas)

# Log-likelihood: sum of log(N(x|mu,sigma)) for each data point
log_like = np.zeros_like(MU)
for i in range(len(sigmas)):
    for j in range(len(mus)):
        log_like[i, j] = np.sum(stats.norm.logpdf(data, MU[i,j], SIGMA[i,j]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Contour of log-likelihood
cf = ax1.contourf(MU, SIGMA, log_like, levels=30, cmap='viridis')
plt.colorbar(cf, ax=ax1)
ax1.scatter(mu_mle, sigma_mle, color='red', s=200, zorder=5, marker='*', label='MLE estimate')
ax1.scatter(true_mu, true_sigma, color='white', s=100, zorder=5, marker='o', label='True params')
ax1.set_xlabel('μ'); ax1.set_ylabel('σ')
ax1.set_title('Log-Likelihood Surface for Normal Distribution')
ax1.legend()

# Data histogram vs fitted distribution
ax2.hist(data, bins=30, density=True, alpha=0.6, color='steelblue', label='Data')
x_plot = np.linspace(data.min(), data.max(), 300)
ax2.plot(x_plot, stats.norm.pdf(x_plot, mu_mle, sigma_mle), 'r-', lw=2, label=f'MLE fit: N({mu_mle:.1f},{sigma_mle:.1f})')
ax2.plot(x_plot, stats.norm.pdf(x_plot, true_mu, true_sigma), 'g--', lw=2, label=f'True: N({true_mu},{true_sigma})')
ax2.set_xlabel('x'); ax2.set_ylabel('Density')
ax2.set_title('MLE-Fitted Distribution vs Data')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 6. Law of Large Numbers Demo

In [ ]:
# As sample size grows, sample mean converges to true mean
true_mean = 3.5  # expected value of a fair die
n_max = 10000
die_rolls = np.random.randint(1, 7, n_max)
running_mean = np.cumsum(die_rolls) / np.arange(1, n_max + 1)

plt.figure(figsize=(10, 5))
plt.plot(running_mean, alpha=0.8, color='steelblue', lw=1.5, label='Running mean')
plt.axhline(true_mean, color='red', ls='--', lw=2, label=f'True mean = {true_mean}')
plt.fill_between(range(n_max),
                 true_mean - 0.5, true_mean + 0.5,
                 alpha=0.1, color='red', label='±0.5 band')
plt.xscale('log')
plt.xlabel('Number of rolls (log scale)')
plt.ylabel('Running mean')
plt.title('Law of Large Numbers: Die Roll Means Converge to 3.5')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

| Concept | Formula | ML Application |
|---------|---------|---------------|
| Conditional probability | P(A\|B) = P(A∩B)/P(B) | Classification, naive Bayes |
| Bayes' theorem | P(A\|B) = P(B\|A)P(A)/P(B) | Bayesian models, medical diagnosis |
| Normal distribution | N(μ, σ²) | Error distribution, feature scaling |
| MLE | argmax_θ Σ log P(xᵢ\|θ) | Training most ML models |
| Expectation | E[X] = Σ x·P(x) | Loss functions, model output |
| Variance | Var(X) = E[(X-μ)²] | Bias-variance tradeoff |

**Next:** [05_statistics_hypothesis.ipynb](05_statistics_hypothesis.ipynb)